In [ ]:
import os

# Physical GPU index. The 5th GPU is index 4.
VISIBLE_GPU = "4"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = VISIBLE_GPU

print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")
print("Restart the kernel after changing VISIBLE_GPU if torch was already imported.")


# Base-State Landscape

This notebook builds a landscape over different full initial states sampled from:

`base_state = model.initialize_state(embedded_inputs)`

So the varying object is the entire recurrent initial state tensor, not just the last position.

Pipeline:
1. sample many full initial states with `initialize_state`
2. compute a 2D PCA plane in that state space
3. evaluate a regular grid on that plane
4. define height from the recurrent residual `f_t - f_{t-1}`

Default height metrics:
- final-step mean residual over positions
- mean residual over both steps and positions


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from transformers import AutoModelForCausalLM, AutoTokenizer

assert transformers.__version__.startswith("4.47."), (
    f"This notebook expects transformers 4.47.x, got {transformers.__version__}. "
    "Switch to the 'huginn-venv' kernel."
)

MODEL_DIR = Path("/data/hansen/serv12/HRM-Deq/models/huginn-0125")
DEVICE = "cuda:0"
DTYPE = torch.bfloat16

QUESTION = (
    "There are 15 trees in the grove. Grove workers will plant trees in the grove today. "
    "After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
)
PROMPT_TEXT = f"Q: {QUESTION}\n\nA:"

NUM_INIT_SAMPLES = 64
GRID_SIZE = 17
NUM_STEPS = 16
ROLLOUT_BATCH_SIZE = 16
INIT_STD_SCALE = 4.0
INIT_STD_OVERRIDE = None
SEED = 7

torch.manual_seed(SEED)
assert MODEL_DIR.exists(), MODEL_DIR


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    device_map={"": DEVICE},
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model.eval()

base_init_std = float(model.config.init_values["std"])
analysis_init_std = float(INIT_STD_OVERRIDE) if INIT_STD_OVERRIDE is not None else base_init_std * INIT_STD_SCALE
analysis_init_scale = 0.0 if base_init_std == 0 else analysis_init_std / base_init_std
analysis_effective_std = analysis_init_std * float(getattr(model, "emb_scale", 1.0))

prompt_ids = tokenizer(PROMPT_TEXT, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
embedded_inputs, block_idx = model.embed_inputs(prompt_ids)

print("prompt length:", prompt_ids.shape[1])
print("embedded_inputs:", tuple(embedded_inputs.shape))
print(f"default init std: {base_init_std:.6f}")
print(f"analysis init std: {analysis_init_std:.6f} (scale={analysis_init_scale:.3f})")
print(f"analysis per-dim std after emb_scale: {analysis_effective_std:.6f}")


In [ ]:
@torch.inference_mode()
def sample_full_base_states(model, embedded_inputs, num_samples, init_scale):
    samples = []
    for _ in range(num_samples):
        state = model.initialize_state(embedded_inputs, scale=init_scale)
        samples.append(state.squeeze(0).detach().cpu().float())
    return torch.stack(samples, dim=0)  # [sample, position, hidden]


full_base_states = sample_full_base_states(model, embedded_inputs, NUM_INIT_SAMPLES, analysis_init_scale)
flat_states = full_base_states.reshape(NUM_INIT_SAMPLES, -1)
center_flat = flat_states.mean(dim=0)
center_state = center_flat.reshape_as(full_base_states[0])

# PCA directions in the space of full base_state tensors.
centered = flat_states - center_flat
U, S, V = torch.pca_lowrank(centered, q=2)
dir1 = V[:, 0]
dir2 = V[:, 1]
sample_coords = centered @ V[:, :2]

radius_x = torch.quantile(sample_coords[:, 0].abs(), 0.95).item()
radius_y = torch.quantile(sample_coords[:, 1].abs(), 0.95).item()

print("full_base_states:", tuple(full_base_states.shape))
print("flat_states:", tuple(flat_states.shape))
print("sample_coords:", tuple(sample_coords.shape))
print("radius_x, radius_y:", radius_x, radius_y)


In [ ]:
x_axis = torch.linspace(-radius_x, radius_x, GRID_SIZE, dtype=torch.float32)
y_axis = torch.linspace(-radius_y, radius_y, GRID_SIZE, dtype=torch.float32)
xx, yy = torch.meshgrid(x_axis, y_axis, indexing="xy")

grid_flat = (
    center_flat.unsqueeze(0)
    + xx.reshape(-1, 1) * dir1.unsqueeze(0)
    + yy.reshape(-1, 1) * dir2.unsqueeze(0)
)
grid_states = grid_flat.reshape(-1, *center_state.shape).to(DTYPE)  # [grid, position, hidden]

print("grid_states:", tuple(grid_states.shape))


In [ ]:
@torch.inference_mode()
def rollout_grid_landscape(model, embedded_inputs, initial_states, block_idx, num_steps, batch_size):
    all_step_residuals = []

    for start in range(0, initial_states.shape[0], batch_size):
        stop = min(start + batch_size, initial_states.shape[0])
        current = initial_states[start:stop].to(DEVICE)
        local_embeds = embedded_inputs.repeat(current.shape[0], 1, 1)
        local_block_idx = block_idx.clone()

        batch_step_residuals = []
        for step in range(num_steps):
            prev = current.clone()
            current, local_block_idx, _ = model.iterate_one_step(
                local_embeds,
                current,
                block_idx=local_block_idx,
                current_step=step,
            )
            delta = current - prev
            delta_norm = torch.linalg.vector_norm(delta.float(), dim=-1)  # [batch, position]
            batch_step_residuals.append(delta_norm.cpu())

        batch_step_residuals = torch.stack(batch_step_residuals, dim=1)  # [batch, step, position]
        all_step_residuals.append(batch_step_residuals)

    step_residuals = torch.cat(all_step_residuals, dim=0)
    return {
        "step_residuals": step_residuals,
        "final_mean_over_positions": step_residuals[:, -1, :].mean(dim=-1),
        "mean_over_steps_and_positions": step_residuals.mean(dim=(1, 2)),
        "final_last_position": step_residuals[:, -1, -1],
    }


landscape = rollout_grid_landscape(
    model=model,
    embedded_inputs=embedded_inputs,
    initial_states=grid_states,
    block_idx=block_idx,
    num_steps=NUM_STEPS,
    batch_size=ROLLOUT_BATCH_SIZE,
)

step_residuals = landscape["step_residuals"]
height_final_mean = landscape["final_mean_over_positions"].reshape(GRID_SIZE, GRID_SIZE).numpy()
height_mean_all = landscape["mean_over_steps_and_positions"].reshape(GRID_SIZE, GRID_SIZE).numpy()
height_last_pos = landscape["final_last_position"].reshape(GRID_SIZE, GRID_SIZE).numpy()

print("step_residuals:", tuple(step_residuals.shape))


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
contour = ax.contourf(xx.numpy(), yy.numpy(), height_final_mean, levels=24, cmap="magma")
ax.scatter(sample_coords[:, 0].numpy(), sample_coords[:, 1].numpy(), s=10, c="white", alpha=0.5)
ax.set_title("Landscape: final-step mean residual over positions")
ax.set_xlabel("PCA direction 1")
ax.set_ylabel("PCA direction 2")
cbar = fig.colorbar(contour, ax=ax)
cbar.set_label("mean_position ||f_T - f_{T-1}||_2")
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
contour = ax.contourf(xx.numpy(), yy.numpy(), height_mean_all, levels=24, cmap="plasma")
ax.scatter(sample_coords[:, 0].numpy(), sample_coords[:, 1].numpy(), s=10, c="white", alpha=0.5)
ax.set_title("Landscape: mean residual over steps and positions")
ax.set_xlabel("PCA direction 1")
ax.set_ylabel("PCA direction 2")
cbar = fig.colorbar(contour, ax=ax)
cbar.set_label("mean_{t,position} ||f_t - f_{t-1}||_2")
fig.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(xx.numpy(), yy.numpy(), height_final_mean, cmap="magma", linewidth=0, antialiased=True)
ax.set_title("3D surface: final-step mean residual")
ax.set_xlabel("PCA direction 1")
ax.set_ylabel("PCA direction 2")
ax.set_zlabel("mean_position ||f_T - f_{T-1}||_2")
fig.tight_layout()
plt.show()


In [ ]:
# Center-point convergence curve on the PCA grid.
center_index = (GRID_SIZE * GRID_SIZE) // 2
center_curve = step_residuals[center_index].mean(dim=-1).numpy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(1, NUM_STEPS + 1), center_curve, marker="o")
ax.set_title("Center-grid convergence curve")
ax.set_xlabel("Step t")
ax.set_ylabel("mean_position ||f_t - f_{t-1}||_2")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()
